# Preparation Huang et al. 2021 data

---

In [ ]:
# Libraries
import os, time, shutil
import numpy as np
import xarray as xr
import rioxarray
from dask.distributed import Lock
from  scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_data =  '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'

---

### Pre-processing

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute', 
                       cores=24, #6
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='00:30:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=5)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)

client

In [ ]:
# Allow large chunks
import dask
dask.config.set(**{'array.slicing.split_large_chunks': True});

In [ ]:
# Function to trim memory of workers
def trim_memory() -> int:
    import ctypes
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

In [ ]:
def prep_huang():

    """
    Prepare Huang data for regridding
    """
    
    def load_data(file): 
    
        # Get data
        ds = xr.open_dataset(
            os.path.join(dir_data, f"huang2021/data_code_to_submit/", file),
            chunks=dict(lat=10000, lon=10000), lock=False)
    
        # Rename lat and lon
        ds = ds.rename(dict(LAT='lat', LON='lon'))
        # Reindex lat
        ds = ds.reindex(lat=ds.lat[::-1])    
        
        return ds
    
    def export_tif(da, file):
        # Export as geotif
        da.rio.to_raster(os.path.join(dir01, file), 
                         tiled=True,
                         lock=Lock("rio"))
    
    # AGB
    ds_agb = load_data('pergridarea_agb.nc')
    da_agb = ds_agb['ASHOOT'].rename('huang_agb')
    export_tif(da_agb, 'huang_agb.tif')
    
    # BGB
    ds_bgb = load_data('pergridarea_bgb.nc')
    da_bgb = ds_bgb['AROOT'].rename('huang_bgb')
    export_tif(da_bgb, 'huang_bgb.tif')
    
    # BGB/ABG ratio
    da_bgbagbratio = (da_bgb / da_agb).rename('huang_bgbagbratio')
    export_tif(da_bgbagbratio, 'huang_bgbagbratio.tif')


%time prep_huang()

In [ ]:
# Close dask cluster
cluster.close()

In [ ]:
# Wait for dask client to completely disconnect
time.sleep(60)

---

### Regridding

In [ ]:
# Import regridding function
from regrid_high_res_v1_01 import regrid_high_res, prep_tif

In [ ]:
def regrid_da(f_source, dir_target, dir_source, dir_out, 
              size_tiles, fill_value=None, olap=1):  
    """Regrid large xarray dataarrays.

    Args:
        f_source (str): The filename (without extension) of the source .tif file to be regridded.
        dir_target (str): Directory containing target grid .tif file.
        dir_source (str): Directory containing the the source  .tif file.
        dir_out (str): Directory to store the output and intermediate files.
        size_tiles (int): Size of the regridding tiles in degrees.
        fill_value (float, optional): Fill value to use in the regridding process. Defaults to None.
        olap (int, optional): Overlap size in degrees for regridding tiles. Defaults to 1.
        
    Returns:
        xarray.Dataset: The combined dataset after regridding.
    """
    # Prepare the target and source data arrays from TIFF files
    da_target = prep_tif(dir_target + 'target_grid_r001.tif', 'target_grid_r001')
    da_source = prep_tif(dir_source + f_source + '.tif', f_source)
    # Regridd source array to target grid
    regrid_high_res(da_target, da_source, dir_out,
                    account='bm0891', partition='shared',
                    size_tiles=size_tiles, olap=olap, fill_value = fill_value,
                    type_export='zarr', del_interm=False)

In [ ]:
for i in ['huang_agb', 'huang_bgb', 'huang_bgbagbratio']:
    %time regrid_da(i, dir01, dir01, dir01, 20, np.nan, 0.2)
# takes about 4min for one file

---

### Fill land nans

In [ ]:
def fill_nans(var, dir_out):
    """
    Fills NaN values in the specified variable's dataset using the nearest valid 
    data, applies a land mask, and exports the result to a new Zarr dataset.

    Args:
        var (str): Name of the variable to process (e.g., 'temperature', 'precipitation').
        dir_out (str): Directory where prepared data is stored and the filled dataset will be exported.

    Returns:
        None
    """

    def fill_nans_array(data, invalid):
        """
        Replace invalid (NaN) data cells by the value of the nearest valid data 
        cell.
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_prep_copernicus_land_mask_r001.zarr')
    var_data_path = os.path.join(dir_out, f'ds_regridded_{var}.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r001.zarr')

    # Read land mask data
    da_land = xr.open_zarr(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .copernicus_land_mask \
                .compute()

    # Read variable data
    da_var = xr.open_zarr(var_data_path)['regridded_' + var]

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled.chunk(dict(lat=5000, lon=5000)) \
             .to_zarr(output_path, mode='w')


for i in ['huang_agb', 'huang_bgb', 'huang_bgbagbratio']:
    %time fill_nans(i, dir01)

---

### Post-processing

In [ ]:
xr.open_mfdataset(dir01 + 'ds_prep_huang_*_r001.zarr', 
                  engine='zarr') \
    .to_zarr(dir01 + 'ds_prep_huang_r001.zarr');

In [ ]:
for i in ['huang_agb', 'huang_bgb', 'huang_bgbagbratio']:
    shutil.rmtree(dir01 + 'ds_prep_' + i + '_r001.zarr')